# 📋 Notebook 1: Business Problem Framing

> **Project:** Financial Fraud Detection on Mobile Money Transactions  
> **Dataset:** PaySim1 (Synthetic Financial Dataset) — Kaggle  
> **Goal:** Identify fraudulent transactions before they complete to minimize financial loss

---

## 🏦 Business Context

Mobile money services process millions of transactions daily across Africa and emerging markets. The PaySim dataset simulates one month of real transaction activity (based on real logs from a mobile money service), providing a realistic fraud detection challenge.

**Key business questions this project answers:**
1. What is the financial impact of fraud in mobile payments?
2. Which transaction types are most vulnerable to fraud?
3. Can we build a model that detects fraud early enough to prevent loss?
4. What is the ROI of deploying such a model?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#0f1117',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'axes.edgecolor': '#444',
    'grid.color': '#2a2a2a',
    'grid.alpha': 0.5,
})

print("Libraries loaded successfully ✓")


## 💰 The Cost of Fraud

Financial fraud has asymmetric costs:
- **Missed fraud (False Negative):** Company absorbs 100% of the fraudulent amount
- **False alarm (False Positive):** Investigation cost + customer friction
- **Correct detection (True Positive):** Small investigation cost, large loss prevented

| Scenario | Cost |
|---|---|
| Miss a $10,000 fraud | -$10,000 |
| Investigate a legitimate transaction | -$50 |
| Block a $10,000 fraud | +$9,950 net |

This asymmetry means we must **prioritize Recall** (catching fraud) even at the cost of some Precision.


In [ ]:
# ── Business Cost Simulation ──────────────────────────────────────────────
np.random.seed(42)

# Simulate 1000 transactions
n_transactions = 1000
fraud_rate = 0.0013
n_fraud = int(n_transactions * fraud_rate * 100)  # scale for illustration
avg_fraud_value = 856_000  # average fraud amount in PaySim

# Scenario comparison
scenarios = {
    'No Detection\n(Status Quo)': {
        'fraud_prevented': 0,
        'false_alarms': 0,
        'cost': -n_fraud * avg_fraud_value
    },
    'Rule-Based\nSystem': {
        'fraud_prevented': 0.30,
        'false_alarms': 200,
        'cost': -n_fraud * avg_fraud_value * 0.70 - 200 * 50
    },
    'ML Model\n(This Project)': {
        'fraud_prevented': 0.89,
        'false_alarms': 45,
        'cost': -n_fraud * avg_fraud_value * 0.11 - 45 * 50
    }
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Business Case: Fraud Detection ROI', fontsize=16, color='white', y=1.02)

# Chart 1: Net cost by scenario
names = list(scenarios.keys())
costs = [s['cost'] for s in scenarios.values()]
colors = ['#E15759', '#F28E2B', '#4CAF50']

bars = axes[0].bar(names, [abs(c) for c in costs], color=colors, width=0.5, edgecolor='white', linewidth=0.5)
axes[0].set_title('Estimated Fraud Loss by Detection Method', color='white', fontsize=12)
axes[0].set_ylabel('Loss (USD)', color='white')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

for bar, cost in zip(bars, costs):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() * 1.02,
                f'${abs(cost)/1e6:.2f}M', ha='center', va='bottom', color='white', fontsize=10)

# Chart 2: Fraud caught %
caught = [0, 30, 89]
axes[1].bar(names, caught, color=colors, width=0.5, edgecolor='white', linewidth=0.5)
axes[1].set_title('% of Fraud Detected', color='white', fontsize=12)
axes[1].set_ylabel('Detection Rate (%)', color='white')
axes[1].set_ylim(0, 110)

for i, v in enumerate(caught):
    axes[1].text(i, v + 2, f'{v}%', ha='center', color='white', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../visuals/01_business_case.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
plt.show()
print("Business case chart saved ✓")


## 🎯 Success Metrics

For this project we define success as:

| Metric | Target | Rationale |
|---|---|---|
| **ROC-AUC** | > 0.97 | Primary metric — handles class imbalance well |
| **Recall (Fraud)** | > 0.85 | Catch most fraud even with more false alarms |
| **Precision (Fraud)** | > 0.80 | Limit investigation fatigue |
| **F1 Score** | > 0.82 | Balanced measure |

## 🚧 Challenges

1. **Class imbalance:** Only ~0.13% of transactions are fraudulent → naive model predicts all non-fraud
2. **Concept drift:** Fraud patterns change over time → model needs regular retraining  
3. **Latency constraints:** Real-time scoring needs < 100ms inference
4. **Interpretability:** Compliance teams need explanations for every declined transaction

## ✅ Summary

This project builds a production-grade ML pipeline to:
- Detect fraudulent TRANSFER and CASH-OUT transactions
- Maximize ROC-AUC while maintaining business-viable recall
- Provide SHAP-based explanations for individual predictions
- Deliver actionable business recommendations for deployment
